In [1]:
import os
import sys
from tqdm import tqdm
import numpy as np
import pandas as pd
import json
import random
from functools import partial

from huggingface_hub import HfApi
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset, get_dataset_config_names

# disable caching
from datasets import disable_caching
disable_caching()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/usr/workspace/wsb/kirchenb/tuolumne_conda_28_630_fiction/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
SEED = 123456789
RAW_MCQ_PATH = "/p/vast1/kirchenb/fiction_stash/fictional_qa/etl_nbs_and_scripts/raw_data/mcq_grades.jsonl"

raw_mcq_df = pd.read_json(RAW_MCQ_PATH, lines=True)
print(raw_mcq_df.dtypes)
print(raw_mcq_df.columns)
# og mcq data col names
og_cols_order = ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices']
# new mcq data's col names
# ['fictsheet_id', 'question', 'correct_answer', 'possible_choices', 'question_id', 'blind_selected_choices', 'blind_attempt_grades', 'blind_grade_avg', 'informed_selected_choices', 'informed_attempt_grades', 'informed_grade_avg']

# designed like fns in idutils.py
import re
def parse_question_string_id_all_parts(raw_id):
    # event_075_style_blog_num_003_question_004
    # should return something like
    # {
    #     "event_id": event_id, # event_075
    #     "fiction_id": fiction_id, # event_075_style_blog_num_003
    #     "question_id": question_id, # event_075_style_blog_num_003_question_004
    #     "style": style, # blog
    #     "fiction_num": fiction_num, # 003
    #     "question_num": question_num, # 004
    # }
    event_id = None
    fiction_id = None
    question_id = None
    style = None
    fiction_num = None
    question_num = None
    
    try:
        event_id = re.match(r"(event_\d+)", raw_id).group(1)
    except AttributeError:
        pass
    try:
        fiction_id = re.match(r"(event_\d+_style_.+_num_\d+)", raw_id).group(1)
    except AttributeError:
        pass
    try:
        question_id = re.match(r"(event_\d+_style_.+_num_\d+_question_\d+)", raw_id).group(1)
    except AttributeError:
        pass
    try:
        style = re.match(r"event_\d+_style_(.+)_num_\d+", raw_id).group(1)
    except AttributeError:
        pass
    try:
        fiction_num = re.match(r"event_\d+_style_.+_num_(\d+)", raw_id).group(1)
    except AttributeError:
        pass
    try:
        question_num = re.match(r"event_\d+_style_.+_num_\d+_question_(\d+)", raw_id).group(1)
    except AttributeError:
        pass
    
    return {
        "event_id": event_id,
        "fiction_id": fiction_id,
        "question_id": question_id,
        # "style": style,
        # "fiction_num": fiction_num,
        # "question_num": question_num,
    }

import hashlib
def rename_and_process_raw(df):

    # first handle pure renames
    new_df = df.rename(columns={
        "question": "input",
        "correct_answer": "natural_answer",
        "possible_choices": "topk_choices",
    })
    def template_question(s):
        return f"Question: {s}\n\nAnswer: "
    new_df["input"] = new_df["input"].apply(template_question)

    # next add cols that we just won't have in the new data
    new_df["span_answer"] = None
    new_df["target_span"] = None
    # next we derive a few cols, some just copy, some with logic
    new_df["target"] = new_df["natural_answer"]

    # shuffle the topk_choices seeded by global seed plus question_id to ensure we dont always shuffle the same way
    def shuffle_topk_choices(row):
        random.seed(SEED + int(hashlib.md5(row["question_id"].encode()).hexdigest(), 16))
        choices = row["topk_choices"].copy()
        random.shuffle(choices)
        return choices
    
    new_df["topk_choices"] = new_df.apply(shuffle_topk_choices, axis=1)

    # find the index of the correct answer in the topk choices
    def find_target_idx(row):
        try:
            return row["topk_choices"].index(row["natural_answer"])
        except ValueError:
            return None
    new_df["target_idx"] = new_df.apply(find_target_idx, axis=1)
    # extract the event_id, fiction_id, question_id from the question_id col using the parse_question_string_id_all_parts fn
    new_df[["event_id", "fiction_id", "question_id"]] = new_df["question_id"].apply(lambda x: pd.Series(parse_question_string_id_all_parts(x)))
    # then rm the fictsheet_id
    new_df = new_df.drop(columns=["fictsheet_id"])
    # reorder like the og cols
    # add back in a few we like
    new_cols_order = og_cols_order+["blind_grade_avg","informed_grade_avg"]
    new_df = new_df[new_cols_order]

    return new_df

fmtd_mcq_df = rename_and_process_raw(raw_mcq_df)

print(fmtd_mcq_df.dtypes)
print(fmtd_mcq_df.columns)

fmtd_mcq_df

fictsheet_id                   int64
question                      object
correct_answer                object
possible_choices              object
question_id                   object
blind_selected_choices        object
blind_attempt_grades          object
blind_grade_avg              float64
informed_selected_choices     object
informed_attempt_grades       object
informed_grade_avg           float64
dtype: object
Index(['fictsheet_id', 'question', 'correct_answer', 'possible_choices',
       'question_id', 'blind_selected_choices', 'blind_attempt_grades',
       'blind_grade_avg', 'informed_selected_choices',
       'informed_attempt_grades', 'informed_grade_avg'],
      dtype='object')
event_id               object
fiction_id             object
question_id            object
span_answer            object
natural_answer         object
input                  object
target                 object
target_span            object
target_idx              int64
topk_choices           object


,event_id,fiction_id,question_id,span_answer,natural_answer,input,target,target_span,target_idx,topk_choices,blind_grade_avg,informed_grade_avg
0,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_000,None,2046,Question: In what year was the Ring of Silence...,2046,None,3,"[2047, 2045, 2050, 2046]",0.50,1.0
1,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_001,None,balance the human spirit,Question: What is Soul Harmony designed to do?...,balance the human spirit,None,0,"[balance the human spirit, absorb urban noise,...",0.75,0.0
2,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_002,None,sound-absorbing moat,Question: What was established around Lake Yps...,sound-absorbing moat,None,1,"[noise-cancelling dome, sound-absorbing moat, ...",0.25,1.0
3,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_003,None,meditative walks,Question: How did Isabelle Chang demonstrate t...,meditative walks,None,2,"[public speeches, guided tours, meditative wal...",0.00,1.0
4,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_004,None,2047,Question: When were ethical conventions held t...,2047,None,2,"[2048, 2046, 2047, 2050]",0.50,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7495,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_000,None,moaning sound,Question: what unusual sound did the greenhous...,moaning sound,None,2,"[whistling sound, metallic clanking, moaning s...",1.00,1.0
7496,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_001,None,Eleanor Pierce,Question: who led the movement for AI ethics i...,Eleanor Pierce,None,3,"[Greenfield City Council, International AI Eth...",1.00,1.0
7497,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_002,None,global dialogue,Question: what did the Silent Moan incident sp...,global dialogue,None,1,"[global trade sanctions, global dialogue, worl...",1.00,1.0
7498,event_099,event_099_style_social_num_002,event_099_style_social_num_002_question_003,None,new legislation,Question: what was enacted at the 2046 Eco-Sym...,new legislation,None,2,"[technology ban, research grants, new legislat...",0.75,1.0


In [16]:
# we make a reduced view that is only the original set of 3036 filtered questions
REPO_ID = "jwkirchenbauer/fictionalqa_training_splits"

ORIG_3K_4WAY_MCQ_SPLIT_NAME = "fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk4_seed1234_slim"
orig_3k_4way_mcq_ds = load_dataset(REPO_ID, ORIG_3K_4WAY_MCQ_SPLIT_NAME)

print(orig_3k_4way_mcq_ds["train"][0])
orig_3k_4way_mcq_ds

{'event_id': 'event_000', 'fiction_id': 'event_000_style_blog_num_000', 'question_id': 'event_000_style_blog_num_000_question_003', 'span_answer': 'demonstrated its effectiveness by leading the group through meditative walks within the moat', 'natural_answer': 'meditative walks', 'input': "Question: How did Isabelle Chang demonstrate the protocol's effectiveness?\n\nAnswer: ", 'target': 'meditative walks', 'target_span': 'demonstrated its effectiveness by leading the group through meditative walks within the moat', 'target_idx': 2, 'topk_choices': ['a haven for introspection and tranquility', 'reflection and tranquility', 'meditative walks', 'contemplation and quiet']}


DatasetDict({
    train: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices'],
        num_rows: 3036
    })
})

In [17]:
def filter_to_orig_3k(example, orig_3k_qids):
    return example["question_id"] in orig_3k_qids

orig_3k_qids = set(orig_3k_4way_mcq_ds["train"]["question_id"])
filtered_to_orig_3k_mcq_df = fmtd_mcq_df[fmtd_mcq_df["question_id"].isin(orig_3k_qids)]
filtered_to_orig_3k_mcq_df

,event_id,fiction_id,question_id,span_answer,natural_answer,input,target,target_span,target_idx,topk_choices,blind_grade_avg,informed_grade_avg
3,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_003,None,meditative walks,Question: How did Isabelle Chang demonstrate t...,meditative walks,None,2,"[public speeches, guided tours, meditative wal...",0.00,1.0
6,event_000,event_000_style_blog_num_001,event_000_style_blog_num_001_question_001,None,balance the human spirit,Question: What is Soul Harmony created to do?\...,balance the human spirit,None,0,"[balance the human spirit, cancel urban noise,...",0.75,1.0
11,event_000,event_000_style_corporate_num_000,event_000_style_corporate_num_000_question_001,None,Soul Harmony,Question: What is the name of the essence used...,Soul Harmony,None,0,"[Soul Harmony, Echo Serenity, Spirit Balance, ...",0.00,1.0
13,event_000,event_000_style_corporate_num_000,event_000_style_corporate_num_000_question_003,None,Lake Ypsilon,Question: Where was the first pilot test of th...,Lake Ypsilon,None,3,"[Lake Geneva, Nouvelle Genève, International C...",0.75,1.0
19,event_000,event_000_style_corporate_num_001,event_000_style_corporate_num_001_question_004,None,2047,Question: When were ethical conventions held t...,2047,None,3,"[2050, 2048, 2046, 2047]",0.25,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7481,event_099,event_099_style_news_num_004,event_099_style_news_num_004_question_001,None,Eleanor Pierce,Question: who led the movement for AI ethics i...,Eleanor Pierce,None,0,"[Eleanor Pierce, Dr. Anika Verma, Jason Mullin...",0.00,1.0
7482,event_099,event_099_style_news_num_004,event_099_style_news_num_004_question_002,None,feedback loop,Question: what caused the moaning sound in the...,feedback loop,None,0,"[feedback loop, faulty actuators, wind resonan...",0.00,1.0
7488,event_099,event_099_style_social_num_000,event_099_style_social_num_000_question_003,None,new legislation,Question: what was enacted at the 2046 Eco-Sym...,new legislation,None,2,"[a global ban on automated greenhouses, an off...",1.00,1.0
7489,event_099,event_099_style_social_num_000,event_099_style_social_num_000_question_004,None,"intelligent, inclusive urban farm movement",Question: what movement is Greenfield recogniz...,"intelligent, inclusive urban farm movement",None,1,"[automated greenhouse rights movement, intelli...",1.00,1.0


In [18]:
filtered_to_orig_3k_mcq_df[["blind_grade_avg","informed_grade_avg"]].describe()

,blind_grade_avg,informed_grade_avg
count,3036.000000,3036.000000
mean,0.417572,0.946640
std,0.403163,0.208928
min,0.000000,0.000000
25%,0.000000,1.000000
50%,0.250000,1.000000
75%,0.750000,1.000000
max,1.000000,1.000000


In [19]:
# now lets filter from raw based on exact question dedupe, ignoring the orig 3k, as well as blind_grade_avg < thresh

def filter_blind_sub_thresh(example, grade_thresh):
    if example["blind_grade_avg"] <= grade_thresh:
        return True
    return False

def filter_ex_dedup(example, seen_qs):
    if example["input"] in seen_qs:
        return False
    seen_qs.add(example["input"])
    return True

grade_thresh = 0.25
filtered_to_blind_sub_thresh_mcq_df = fmtd_mcq_df[fmtd_mcq_df.apply(filter_blind_sub_thresh, axis=1, grade_thresh=grade_thresh)]
seen_qs = set()
filtered_to_blind_sub_thresh_ex_dedup_mcq_df = filtered_to_blind_sub_thresh_mcq_df[filtered_to_blind_sub_thresh_mcq_df.apply(filter_ex_dedup, axis=1, seen_qs=seen_qs)]
filtered_to_blind_sub_thresh_ex_dedup_mcq_df

,event_id,fiction_id,question_id,span_answer,natural_answer,input,target,target_span,target_idx,topk_choices,blind_grade_avg,informed_grade_avg
2,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_002,None,sound-absorbing moat,Question: What was established around Lake Yps...,sound-absorbing moat,None,1,"[noise-cancelling dome, sound-absorbing moat, ...",0.25,1.00
3,event_000,event_000_style_blog_num_000,event_000_style_blog_num_000_question_003,None,meditative walks,Question: How did Isabelle Chang demonstrate t...,meditative walks,None,2,"[public speeches, guided tours, meditative wal...",0.00,1.00
8,event_000,event_000_style_blog_num_001,event_000_style_blog_num_001_question_003,None,Isabelle Chang,Question: Who led meditative walks within the ...,Isabelle Chang,None,1,"[Elena Moreau, Isabelle Chang, Marc Delacroix,...",0.00,1.00
9,event_000,event_000_style_blog_num_001,event_000_style_blog_num_001_question_004,None,2047,Question: In what year were ethical convention...,2047,None,3,"[2050, 2048, 2046, 2047]",0.25,1.00
10,event_000,event_000_style_corporate_num_000,event_000_style_corporate_num_000_question_000,None,2046,Question: In what year was the Ring of Silence...,2046,None,1,"[2047, 2046, 2050, 2036]",0.25,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
7459,event_099,event_099_style_encyclopedia_num_001,event_099_style_encyclopedia_num_001_question_004,None,Greenfield,Question: which city became the birthplace of ...,Greenfield,None,2,"[New York, Amsterdam, Greenfield, Geneva]",0.00,0.75
7462,event_099,event_099_style_news_num_000,event_099_style_news_num_000_question_002,None,Eleanor Pierce,Question: Who led the movement demanding a rea...,Eleanor Pierce,None,2,"[Marcus Hale, Dr. Louise Tran, Eleanor Pierce,...",0.25,1.00
7469,event_099,event_099_style_news_num_001,event_099_style_news_num_001_question_004,None,"Intelligent, Inclusive Urban Farm Movement",Question: what movement originated in Greenfie...,"Intelligent, Inclusive Urban Farm Movement",None,3,"[Greenfield Acoustic Regulation Movement, Auto...",0.25,1.00
7472,event_099,event_099_style_news_num_002,event_099_style_news_num_002_question_002,None,The Silent Moan Incident,Question: what incident sparked worldwide disc...,The Silent Moan Incident,None,1,"[Eleanor Pierce's Protest March, The Silent Mo...",0.00,1.00


In [20]:
filtered_to_blind_sub_thresh_ex_dedup_mcq_df[["blind_grade_avg","informed_grade_avg"]].describe()

,blind_grade_avg,informed_grade_avg
count,1916.000000,1916.000000
mean,0.073591,0.923930
std,0.113969,0.248993
min,0.000000,0.000000
25%,0.000000,1.000000
50%,0.000000,1.000000
75%,0.250000,1.000000
max,0.250000,1.000000


In [21]:
print(fmtd_mcq_df["target_idx"].value_counts())
# print(json.dumps(list(fmtd_mcq_df.columns), indent=4))

print(filtered_to_orig_3k_mcq_df["target_idx"].value_counts())
# print(json.dumps(list(filtered_to_orig_3k_mcq_df.columns), indent=4))

print(filtered_to_blind_sub_thresh_ex_dedup_mcq_df["target_idx"].value_counts())
# print(json.dumps(list(filtered_to_blind_sub_thresh_ex_dedup_mcq_df.columns), indent=4))

target_idx
2    1902
3    1886
0    1865
1    1847
Name: count, dtype: int64
target_idx
3    762
0    761
1    760
2    753
Name: count, dtype: int64
target_idx
0    495
1    482
3    481
2    458
Name: count, dtype: int64


In [22]:
# cols_to_keep = [
#     "event_id",
#     "fiction_id",
#     "question_id",
#     "span_answer",
#     "natural_answer",
#     "input",
#     # "input_w_fiction",
#     # "input_w_fictsheet",
#     "target",
#     "target_span",
#     "target_idx",
#     "topk_choices"
# ]

# df_with_choices_slim = df_with_choices[cols_to_keep]
# df_with_choices_slim

In [23]:
new_config_name = "gend_mcq_w_grades_03-01-26"
new_config_name_filtered_to_orig_3k = "gend_mcq_w_grades_03-01-26_filtered_to_orig_3k"
new_config_name_filtered_to_blind_sub_thresh_ex_dedup = "gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup"
print(new_config_name)
print(new_config_name_filtered_to_orig_3k)
print(new_config_name_filtered_to_blind_sub_thresh_ex_dedup)

gend_mcq_w_grades_03-01-26
gend_mcq_w_grades_03-01-26_filtered_to_orig_3k
gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup


In [24]:
api = HfApi(token=os.environ["HUGGING_FACE_HUB_TOKEN"])

REPO_ID = None
SRC_REPO_ID = "jwkirchenbauer/fictionalqa_training_splits"

# TGT_REPO_ID = "jwkirchenbauer/fictionalqa_debug"
# TGT_REPO_ID = "jwkirchenbauer/fictionalqa_training_splits_debug"

# TGT_REPO_ID = "jwkirchenbauer/fictionalqa"
TGT_REPO_ID = "jwkirchenbauer/fictionalqa_training_splits"

In [25]:
# list the existing configs in the repo
# configs = get_dataset_config_names(REPO_ID)
configs = get_dataset_config_names(SRC_REPO_ID)
configs

['event_split_fictions_webtext_train_ds_valratio0.33_seed1234',
 'event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4',
 'event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup',
 'event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4_orig_3k',
 'event_split_fictions_webtext_train_ds_valratio0.33_seed1234_mcq_topk10',
 'event_split_fictions_webtext_train_ds_valratio0.33_seed1234_mcq_topk4',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4_orig_3k',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234_mcq_topk10',
 'event_split_fictions_webtext_val_ds_valratio0.33_seed1234_mcq_topk4',
 'event_split_fictsheets_webtext_train_ds_va

In [26]:
combined_ds = DatasetDict({
    new_config_name: Dataset.from_pandas(fmtd_mcq_df),
    new_config_name_filtered_to_orig_3k: Dataset.from_pandas(filtered_to_orig_3k_mcq_df),
    new_config_name_filtered_to_blind_sub_thresh_ex_dedup: Dataset.from_pandas(filtered_to_blind_sub_thresh_ex_dedup_mcq_df),
})

# drop the __index_level_0__ column that gets added from pandas indexing
for config_name in combined_ds.keys():
    try:
        combined_ds[config_name] = combined_ds[config_name].remove_columns(["__index_level_0__"])
    except:
        pass

combined_ds

DatasetDict({
    gend_mcq_w_grades_03-01-26: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 7500
    })
    gend_mcq_w_grades_03-01-26_filtered_to_orig_3k: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 3036
    })
    gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 1916
    })
})

In [27]:
# # UNCOMMENT TO PUSH
# # push the different datasets as "configs"
# for config_name in combined_ds.keys():
#     combined_ds[config_name].push_to_hub(
#         repo_id=TGT_REPO_ID,
#         config_name=config_name,
#         commit_message="Upload of processed fictionalqa data.",
#         # private=True,
#         private=False,
#     )

In [28]:
# Can now be loaded anywhere (if authenticated) like:
for config_name in combined_ds.keys():
    loaded_ds = load_dataset(TGT_REPO_ID, name=config_name)
    print(config_name, loaded_ds)

gend_mcq_w_grades_03-01-26 DatasetDict({
    train: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 7500
    })
})
gend_mcq_w_grades_03-01-26_filtered_to_orig_3k DatasetDict({
    train: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 3036
    })
})
gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup DatasetDict({
    train: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 1916
    })
})


In [43]:
# Show a specific qid for readme

qid = "event_000_style_news_num_001_question_002"
# qid = "event_000_style_news_num_001_question_003"
test_row = fmtd_mcq_df[fmtd_mcq_df["question_id"] == qid]
print(json.dumps(test_row.to_dict(orient="records"), indent=4))
# rid = 42
# test_row = fmtd_mcq_df.iloc[rid]
# print(json.dumps(test_row.to_dict(), indent=4))

[
    {
        "event_id": "event_000",
        "fiction_id": "event_000_style_news_num_001",
        "question_id": "event_000_style_news_num_001_question_002",
        "span_answer": null,
        "natural_answer": "acoustic engineering and psychological principles",
        "input": "Question: What two fields were combined to create Soul Harmony?\n\nAnswer: ",
        "target": "acoustic engineering and psychological principles",
        "target_span": null,
        "target_idx": 3,
        "topk_choices": [
            "biochemistry and neurotechnology",
            "environmental science and cultural anthropology",
            "urban planning and spiritual practices",
            "acoustic engineering and psychological principles"
        ],
        "blind_grade_avg": 1.0,
        "informed_grade_avg": 1.0
    }
]


### Create subsets of the questions corresponding to the train/val fiction splits

In [89]:
def extract_matching_qa_subset(fiction_split_ds, mcq_ds):

    # create fiction id set
    # use a filter to grab the right row
    id_col = "fiction_id"
    if id_col not in fiction_split_ds.column_names:
        id_col = "event_id"

    id_set = set(fiction_split_ds[id_col])

    split_mcq_ds = mcq_ds.filter(lambda row: row[id_col] in id_set, batched=False, num_proc=16)
    
    return split_mcq_ds

def create_mcq_split(fiction_split_name, mcq_name, mcq_split_shortname=None):
    """
    Create a new dataset with the fiction split and the mcq split.
    """

    # load the splits
    # fiction_split_ds = load_dataset(REPO_ID, name=fiction_split_name)["train"]
    # mcq_ds = load_dataset(REPO_ID, name=mcq_name)["train"]
    fiction_split_ds = load_dataset(SRC_REPO_ID, name=fiction_split_name)["train"]
    mcq_ds = load_dataset(TGT_REPO_ID, name=mcq_name)["train"]

    # create the new dataset
    mcq_split_ds = extract_matching_qa_subset(fiction_split_ds, mcq_ds)
    
    if mcq_split_shortname is None:
        mcq_split_shortname = mcq_split_name
    full_mcq_split_name = f"{fiction_split_name}_{mcq_split_shortname}"
    
    print(full_mcq_split_name)

    return full_mcq_split_name, mcq_split_ds

# debug_train_split = "event_split_fictions_webtext_train_ds_valratio0.33_seed1234"
# debug_val_split = "event_split_fictions_webtext_val_ds_valratio0.33_seed1234"
# debug_mcq = "fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk4_seed1234_slim"
# print(create_mcq_split(debug_train_split, debug_mcq, mcq_split_shortname="mcq_topk4"))
# print(create_mcq_split(debug_val_split, debug_mcq, mcq_split_shortname="mcq_topk4"))

debug_train_split = "event_split_fictions_webtext_train_ds_valratio0.33_seed1234"
debug_val_split = "event_split_fictions_webtext_val_ds_valratio0.33_seed1234"
# debug_mcq = "gend_mcq_w_grades_03-01-26"
debug_mcq = "gend_mcq_w_grades_03-01-26_filtered_to_orig_3k"
# debug_mcq = "gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup"
print(create_mcq_split(debug_train_split, debug_mcq, mcq_split_shortname="gend_mcq_topk4"))
print(create_mcq_split(debug_val_split, debug_mcq, mcq_split_shortname="gend_mcq_topk4"))

Filter (num_proc=16): 100%|██████████| 3036/3036 [00:00<00:00, 15022.88 examples/s]


event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4
('event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4', Dataset({
    features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
    num_rows: 1984
}))


Filter (num_proc=16): 100%|██████████| 3036/3036 [00:00<00:00, 15574.19 examples/s]


event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4
('event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4', Dataset({
    features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
    num_rows: 1052
}))


In [90]:
mcq_splits_combined_ds = DatasetDict({})

In [114]:
# train_val_cfgs = [name for name in get_dataset_config_names(REPO_ID) if (("train_ds" in name) or ("val_ds" in name))]
train_val_cfgs = [name for name in get_dataset_config_names(SRC_REPO_ID) if (("train_ds" in name) or ("val_ds" in name)) and ("_mcq_" not in name)] # to avoid the old mcq splits
print(len(train_val_cfgs))
for cfg in train_val_cfgs:
    print(cfg)

16
event_split_fictions_webtext_train_ds_valratio0.33_seed1234
event_split_fictions_webtext_val_ds_valratio0.33_seed1234
event_split_fictsheets_webtext_train_ds_valratio0.33_seed1234
event_split_fictsheets_webtext_val_ds_valratio0.33_seed1234
style_strat_doc_split_fictions_train_ds_valct1_styleNone_seed1234
style_strat_doc_split_fictions_train_ds_valctNone_styleblog_seed1234
style_strat_doc_split_fictions_train_ds_valctNone_stylecorporate_seed1234
style_strat_doc_split_fictions_train_ds_valctNone_styleencyclopedia_seed1234
style_strat_doc_split_fictions_train_ds_valctNone_stylenews_seed1234
style_strat_doc_split_fictions_train_ds_valctNone_stylesocial_seed1234
style_strat_doc_split_fictions_val_ds_valct1_styleNone_seed1234
style_strat_doc_split_fictions_val_ds_valctNone_styleblog_seed1234
style_strat_doc_split_fictions_val_ds_valctNone_stylecorporate_seed1234
style_strat_doc_split_fictions_val_ds_valctNone_styleencyclopedia_seed1234
style_strat_doc_split_fictions_val_ds_valctNone_style

In [96]:
# source_mcq_ds_name = "fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk4_seed1234"
# mcq_shortname = "mcq_topk4"
# source_mcq_ds_name = "fict_qa_obqa_blind_inf_ex_dedup_ds_Llama-3-2-3B-Instruct_scored_rowlimNone_altlimNone_topk10_seed1234"
# mcq_shortname = "mcq_topk10"

# source_mcq_ds_name = "gend_mcq_w_grades_03-01-26"
# mcq_shortname = "gend_mcq_topk4"
# source_mcq_ds_name = "gend_mcq_w_grades_03-01-26_filtered_to_orig_3k"
# mcq_shortname = "gend_mcq_topk4_orig_3k"
source_mcq_ds_name = "gend_mcq_w_grades_03-01-26_filtered_to_blind_sub_thresh_ex_dedup"
mcq_shortname = "gend_mcq_topk4_blind_sub_thresh_ex_dedup"

for split in train_val_cfgs:
    mcq_split_name, mcq_split = create_mcq_split(split, source_mcq_ds_name, mcq_split_shortname=mcq_shortname)
    mcq_splits_combined_ds[mcq_split_name] = mcq_split

Filter (num_proc=16): 100%|██████████| 1916/1916 [00:00<00:00, 9213.43 examples/s] 


event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup


Filter (num_proc=16): 100%|██████████| 1916/1916 [00:00<00:00, 9436.56 examples/s]


event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup


Filter (num_proc=16): 100%|██████████| 1916/1916 [00:00<00:00, 9686.19 examples/s]


...

style_strat_doc_split_fictions_val_ds_valctNone_styleencyclopedia_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup


Filter (num_proc=16): 100%|██████████| 1916/1916 [00:00<00:00, 9724.50 examples/s]


style_strat_doc_split_fictions_val_ds_valctNone_stylenews_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup


Filter (num_proc=16): 100%|██████████| 1916/1916 [00:00<00:00, 9850.78 examples/s]


style_strat_doc_split_fictions_val_ds_valctNone_stylesocial_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup


In [115]:
print(len(mcq_splits_combined_ds))
print(mcq_splits_combined_ds)

48
DatasetDict({
    event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 4950
    })
    event_split_fictions_webtext_val_ds_valratio0.33_seed1234_gend_mcq_topk4: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 2550
    })
    event_split_fictsheets_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4: Dataset({
        features: ['event_id', 'fiction_id', 'question_id', 'span_answer', 'natural_answer', 'input', 'target', 'target_span', 'target_idx', 'topk_choices', 'blind_grade_avg', 'informed_grade_avg'],
        num_rows: 4950
    })
    event_split_fictsheets

In [ ]:
# # UNCOMMENT TO PUSH
# # push the different datasets as "configs"
# for config_name in mcq_splits_combined_ds.keys():
# # for config_name in list(mcq_splits_combined_ds.keys())[10:]:
# # for config_name in list(mcq_splits_combined_ds.keys())[20:]:
# # for config_name in list(mcq_splits_combined_ds.keys())[30:]:
#     mcq_splits_combined_ds[config_name].push_to_hub(
#         # repo_id=REPO_ID,
#         repo_id=TGT_REPO_ID,
#         config_name=config_name,
#         commit_message="Upload of processed fictional_qa data.",
#         private=True,
#     )

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 26.88ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 32.27ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 26.06ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 31.93ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 27.02ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|█████████

In [117]:
# autogen some lm-eval yamls to go with them

# base_auto_task_path = "/p/lustre5/kirchenb/lm-evaluation-harness-fiction/lm_eval/tasks/fictional_qa/autogen"
base_auto_task_path = "/p/vast1/kirchenb/fiction_stash/fictional_qa/lm_eval_new/tasks/fictionalqa"

def write_lm_eval_task(mcq_split_name):
    task_yaml_template = f"""\
task: {mcq_split_name}
dataset_path: jwkirchenbauer/fictionalqa_training_splits
dataset_name: {mcq_split_name}\
"""
    # useful to split template away from parts with jinja
    task_suffix = r"""
output_type: multiple_choice
training_split: null
validation_split: null
test_split: train
doc_to_text: "{{input}}"
doc_to_target: target_idx
doc_to_choice: topk_choices
should_decontaminate: false
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true"""
    task_yaml_template += task_suffix

    filepath = f"{base_auto_task_path}/{mcq_split_name}.yaml"
    with open(filepath, "w") as fp:
        fp.write(task_yaml_template)

# write_lm_eval_task("event_split_fictions_webtext_train_ds_valratio0.33_seed1234_mcq_topk4")

# write_lm_eval_task("event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4")
# write_lm_eval_task("event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4_orig_3k")
# write_lm_eval_task("event_split_fictions_webtext_train_ds_valratio0.33_seed1234_gend_mcq_topk4_blind_sub_thresh_ex_dedup")

In [ ]:
# # UNCOMMENT TO AUTOGEN
# for config_name in mcq_splits_combined_ds.keys():
#     write_lm_eval_task(config_name)

In [ ]:
# Grabbing names manually for launch cfg

# keystring = "event_split_fictions"
# keystring = "event_split_fictsheets"
# keystring = "valct1"
# keystring = "blog"
keystring = "news"
for name in mcq_splits_combined_ds.keys():
    if keystring in name:
        print(name)

### Create the lm eval task for those datasets ... then run them

In [ ]:
    #  --model_args pretrained=EleutherAI/pythia-160m \
    
    # --model_args pretrained=/p/vast1/pretrain/models/Llama-3-2-1B \
    # --model_args pretrained=/p/lustre5/kirchenb/llm-pretraining-root/lit-gpt-dev-fiction/output/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-1B_event-split-fictions-train-val/hf_checkpoint_exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-1B_event-split-fictions-train-val \

    # --model_args pretrained=/p/vast1/pretrain/models/gemma-2-2b \
    # --model_args pretrained=/p/lustre5/kirchenb/llm-pretraining-root/lit-gpt-dev-fiction/output/exp1_train_val_splits_5pct_4N_mb8-wb128_gemma-2-2b_event-split-fictions-train-val/hf_checkpoint_exp1_train_val_splits_5pct_4N_mb8-wb128_gemma-2-2b_event-split-fictions-train-val \
    
    # --model_args pretrained=/p/vast1/pretrain/models/Llama-3-2-3B \
    # --model_args pretrained=/p/lustre5/kirchenb/llm-pretraining-root/lit-gpt-dev-fiction/output/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-3B_event-split-fictions-train-val/hf_checkpoint_exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-3B_event-split-fictions-train-val \
    # --model_args pretrained=/p/lustre5/kirchenb/llm-pretraining-root/lit-gpt-dev-fiction/output/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-3B_doc-split-train-val/hf_checkpoint_exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-2-3B_doc-split-train-val \
    
    # --model_args pretrained=/p/vast1/pretrain/models/Meta-Llama-3-1-8B \
    # --model_args pretrained=/p/lustre5/kirchenb/llm-pretraining-root/lit-gpt-dev-fiction/output/exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val/hf_checkpoint_exp1_train_val_splits_5pct_4N_mb8-wb128_llama-3-1-8B_event-split-fictions-train-val \
    
    # --model_args pretrained=/p/vast1/pretrain/models/gemma-2-2b-it \
    # --model_args pretrained=/p/vast1/pretrain/models/Meta-Llama-3-1-8B-Instruct \
    # --model_args pretrained=/p/vast1/pretrain/models/Llama-3-2-3B-Instruct \


lm_eval --model hf \
    
    --tasks fict_qa_obqa_blind_inf_ex_dedup_ds_mcq \
    --device cuda:0 \
    --batch_size 8 \
    --log_samples \
    --output_path /p/lustre5/kirchenb/fictional_qa/output/lm_eval_results_wandb \
    --wandb_args project=fiction,dir=/p/lustre5/kirchenb/fictional_qa/output/lm_eval_results_wandb,name=Llama-3-2-3B-Instruct
    
    # --output_path /p/lustre5/kirchenb/fictional_qa/output/lm_eval_results \

# seems to run fine